<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [5]</a>'.</span>

# Startup Success Forecasting

Generate a forecasting dataset about startup outcomes using the LightningRod SDK.

This notebook sources news articles about startup funding rounds, acquisitions, shutdowns, IPOs, and growth milestones from July 2024 to February 2026. It uses `NewsSeedGenerator` with a set of targeted search queries to cover the full range of startup success and failure signals.

The generated questions can be used to train or evaluate models that forecast startup outcomes — useful for investors, analysts, and researchers studying the venture ecosystem.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from datetime import datetime

from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Seed Sourcing

We use `NewsSeedGenerator` to pull articles from Google News covering the 18-month window from July 2024 to February 2026. Articles are fetched in bi-weekly batches (`interval_duration_days=14`) to ensure even temporal coverage across the period.

Five search queries target distinct startup outcome dimensions:

| Query | Dimension |
|---|---|
| Startup funding rounds Series A B C | Early and growth-stage fundraising |
| Startup late-stage funding valuation unicorn | Late-stage and unicorn milestones |
| Startup acquisition merger deal | M&A outcomes |
| Startup shutdown failure bankruptcy | Negative outcomes |
| Startup IPO public listing valuation | Exit via public markets |
| Startup revenue growth milestone ARR | Revenue and traction signals |

Each query fetches up to 15 articles per 14-day interval, keeping the iteration scope manageable while giving broad topical coverage.

In [3]:
from lightningrod import NewsSeedGenerator

seed_generator = NewsSeedGenerator(
    start_date=datetime(2024, 7, 1),
    end_date=datetime(2026, 2, 1),
    interval_duration_days=14,  # bi-weekly batches across the 18-month window
    articles_per_search=15,     # articles per query per interval
    search_query=[
        "startup funding rounds Series A B C venture capital",
        "startup late-stage funding valuation unicorn",
        "startup acquisition merger deal",
        "startup shutdown failure bankruptcy",
        "startup IPO public listing valuation",
        "startup revenue growth milestone ARR",
    ],
)

## Pipeline Configuration

This pipeline generates binary (yes/no) forecasting questions about startup outcomes
from startup news articles. Each seed article produces up to 5 questions spanning five
outcome dimensions:

- **Follow-on funding** — Will [Startup] raise a Series B/C by [date]?
- **Acquisition** — Will [Startup] be acquired by [date]?
- **Survival** — Will [Startup] still be operating by [date]?
- **Valuation milestone** — Will [Startup] reach a $Xbn valuation by [date]?
- **Revenue/growth milestone** — Will [Startup] reach $XM ARR or X users by [date]?

Questions are generated by a `ForwardLookingQuestionGenerator` and filtered by a
`FilterCriteria` rubric (min_score=0.7) to keep only specific, verifiable questions.
A `NewsContextGenerator` enriches each question with fresh web context before the
`WebSearchLabeler` resolves the yes/no label.

In [4]:
from lightningrod import (
    BinaryAnswerType,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
    FilterCriteria,
)

# Answer type
answer_type = BinaryAnswerType()

# Instructions and examples for the question generator
instructions = """
Generate binary (yes/no) forecasting questions about startup outcomes.
Questions must be specific, self-contained, and verifiable via web search.
Always include the startup name and a concrete date or numeric threshold.
Cover both positive outcomes (fundraising, growth, IPO) and negative/failure
outcomes (shutdown, acqui-hire, missed milestones) to ensure diversity.
Horizon: the outcome should be resolvable within 3-12 months of the question date.
Criteria: binary outcome, exact dates or thresholds, startup name present,
forward-looking (not asking about past events), verifiable via public web search.
"""

good_examples = [
    "Will Anthropic raise a Series E funding round by June 2025?",
    "Will Bolt Financial complete its planned IPO by Q3 2025?",
    "Will Hopin (the events startup) be acquired or shut down by January 2025?",
    "Will Harvey AI reach a $1B valuation by mid-2025?",
    "Will Deel reach $500M ARR by end of 2025?",
    "Will Stripe go public via IPO or direct listing by December 2025?",
    "Will Klarna complete its US IPO by September 2025?",
    "Will Stability AI cease operations or be acquired by July 2025?",
    "Will Mistral AI close a Series B funding round by April 2025?",
    "Will Perplexity AI reach 10 million daily active users by Q2 2025?",
]

bad_examples = [
    "Will this startup succeed? (too vague, no startup name)",
    "Is [Startup] a good investment? (not verifiable, subjective)",
    "What happened to [Startup]? (backward-looking)",
    "Will the startup do well? (no concrete threshold or date)",
    "Will [Startup] grow? (no numeric target or deadline)",
]

# Question generator with quality filter
question_generator = ForwardLookingQuestionGenerator(
    instructions=instructions,
    examples=good_examples,
    bad_examples=bad_examples,
    filter_=FilterCriteria(
        rubric=(
            "The question must name a specific startup, include a concrete date or "
            "numeric threshold, be forward-looking, and be verifiable via web search."
        ),
        min_score=0.7,
    ),
    answer_type=answer_type,
    questions_per_seed=5,
)

# Context generator — fetches fresh news for each question before labeling
context_generators = [
    NewsContextGenerator(
        articles_per_query=3,
        num_search_queries=1,
        num_articles=5,
    )
]

# Labeler — resolves yes/no via web search
labeler = WebSearchLabeler(answer_type=answer_type)

# Full pipeline
pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    context_generators=context_generators,
    labeler=labeler,
)

## Run (Demo)

Run the pipeline with a small `max_questions` cap for a quick smoke-test.
Increase `MAX_QUESTIONS` (or remove the cap entirely) for a full production run.

The call blocks until all questions are generated and labeled — typically a few minutes
for 10 questions, longer for larger runs.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [5]:
MAX_QUESTIONS = 10  # Increase for a full run (e.g. 500 or 1000)

dataset = lr.transforms.run(
    pipeline,
    max_questions=MAX_QUESTIONS,
    name="Startup Forecasting Demo",
)

samples = dataset.download()
pct_valid = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples generated ({pct_valid:.1f}% valid)")

/Users/bart/.pyenv/versions/3.11.2/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Job Failed                                                                                                  │
│                                                                                                                 │
│  Job completed with 0 valid rows (1 total samples, all invalid/errors)                                          │
│                                                                                                                 │
│  This typically happens when:                                                                                   │
│    • Filter criteria is too strict                                                                              │
│    • Labeling failed (e.g., questions couldn't be answered or had low confidence)                               │
│    • Seed generation found no suitable content                                                                  │
│                                                                                                                 │
│  Next steps:                                                                                                    │
│    • Check the dataset samples to see specific failure reasons in the 'meta.filter_reason' field                │
│    • Adjust and retry the transform pipeline (e.g., lower confidence thresholds, relax filter criteria)         │
│    • If the problem persists, contact support or open a GitHub issue:                                           │
│  ]8;id=400964;https://github.com/lightning-rod-labs/lightningrod-python-sdk/issues\https://github.com/lightning-rod-labs/lightningrod-python-sdk/issues]8;;\                                           │
│                                                                                                                 │
│    Total cost: $0.00                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exception: Transform job ba66eab7-0c1c-4711-afcb-317b8667c4e6 failed: Job completed with 0 valid rows (1 total samples, all invalid/errors)

## View Results

Inspect the generated questions and labels.

In [ ]:
import pandas as pd

rows = dataset.flattened(answer_type)
df = pd.DataFrame(rows)

print(f"Total rows: {len(df)}")

display_cols = ["question_text", "answer", "label_confidence", "is_valid", "invalid_reason"]
df[[c for c in display_cols if c in df.columns]]